# Code de référence pour être utiliser

In [1]:
import glob
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import subprocess
from io import StringIO
import shutil
import re
import warnings

In [ ]:
# ===============================
# LISTE DES CAS
# ===============================

CAS_LIST = [
    "Rainwater-1",
    "Rainwater-2",
    "Seawater-1",
    "Seawater-2",
    "Seawater-3",
    "Surface-Complexation-1",
    "Cation-Exchange-1",
    "Cation-Exchange-2",
    "1D-Very-Simple"
]

# ===============================
# DOSSIERS DE RECHERCHE
# ===============================

start_dirs = [
    Path.home(),
    Path.home() / "Desktop",
    Path.home() / "Documents",
    Path.home() / "Downloads"
]

# ===============================
# RECHERCHE DES DOSSIERS CAS
# ===============================

def find_case_dir(case_name):
    for start in start_dirs:
        for candidate in start.rglob(case_name):
            if candidate.is_dir():
                return candidate
    return None

CASE_DIRS = {}

for case in CAS_LIST:

    base_dir = find_case_dir(case)

    if base_dir is None:
        print(f"{case} introuvable")
        continue

    missing = [
        f for f in ["clean.bat", "run_1proc.bat"]
        if not (base_dir / f).exists()
    ]

    if missing:
        print(f"{case} trouvé mais fichiers manquants : {missing}")
        continue

    CASE_DIRS[case] = base_dir
    print(f"✔ {case} trouvé : {base_dir}")

print("Tous les clean.bat et run_1proc.bat sont présents")


In [ ]:
#===============================
# CLEAN
# ===============================

def run_clean(case_name, case_dir):

    print(f"\n Nettoyage du cas : {case_name}")
    print(f" Dossier : {case_dir}")

    clean_bat = case_dir / "clean.bat"

    print(f"▶ Exécution : {clean_bat}")

    try:

        subprocess.run(
            ["cmd.exe", "/c", str(clean_bat)],
            cwd=str(case_dir),
            timeout=3,
            check=True
        )

    except subprocess.TimeoutExpired:
        print(f" TIMEOUT clean (3s) : {case_name}")
        return

    out_files = list(case_dir.glob("*.out"))

    if out_files:
        print(" Nettoyage incomplet, fichiers .out restants :")
        for f in out_files:
            print("   -", f.name)

    else:
        print(" Nettoyage validé : aucun fichier .out")

print("\n=== Début des clean ===")

for case_name, case_dir in CASE_DIRS.items():
    run_clean(case_name, case_dir)

print("\n=== Tous les clean terminés ===")

In [ ]:
# ===============================
# RUN CRUNCHODITI
# ===============================

def run_case(case_name, case_dir):

    print(f"\nCalcul CrunchODiTi : {case_name}")
    print(f" Dossier : {case_dir}")

    run_bat = case_dir / "run_1proc.bat"

    print(f"▶ Exécution : {run_bat}")

    try:

        subprocess.run(
            ["cmd.exe", "/c", str(run_bat)],
            cwd=str(case_dir),
            timeout=7,
            check=True
        )

    except subprocess.TimeoutExpired:
        print(f" TIMEOUT run (3s) : {case_name}")
        return

    out_files = list(case_dir.glob("*.out"))

    if not out_files:
        print(" Aucun fichier .out généré")
    else:
        print(" Calcul terminé, fichiers .out générés :")
        for f in out_files:
            print("   -", f.name)

print("\n=== DÉBUT DES RUN ===")

for case_name, case_dir in CASE_DIRS.items():
    run_case(case_name, case_dir)

print("\n=== TOUS LES RUN TERMINÉS ===")

In [ ]:
# ===============================
# EXPORT RESULTATS
# ===============================

OUT_DIR = Path.home() / "Downloads" / "resultat_CrunchODITI"
OUT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")

print("\n=== COPIE DES RESULTATS ===")

for case_name, base_dir in CASE_DIRS.items():

    case_name_norm = case_name.lower().replace("-", "_")

    case_out_dir = OUT_DIR / case_name_norm
    case_out_dir.mkdir(exist_ok=True)

    # fichiers .in
    in_files = list(base_dir.glob("*.in"))

    # fichiers résultats tabulated
    tabulated_files = list(base_dir.glob("*_tabulated.out"))

    files_found = in_files + tabulated_files

    # ===============================
    # CAS SPECIAL : 1D-Very-Simple
    # ===============================

    if case_name == "1D-Very-Simple":

        xyz_files = list(base_dir.glob("proc_0Crunchfile.xyz.*"))

        if xyz_files:

            # choisir celui avec le numéro le plus élevé
            latest_xyz = max(
                xyz_files,
                key=lambda f: int(f.name.split(".")[-1])
            )

            files_found.append(latest_xyz)

    # ===============================

    if not files_found:
        print(f"{case_name} : aucun fichier trouvé")
        continue

    for src in files_found:

        dst = case_out_dir / f"{case_name_norm}_{timestamp}_{src.name}"

        dst.write_text(src.read_text(encoding="utf-8", errors="ignore"))

        print(f"✔ {case_name} → {dst}")